In [1]:
import numpy as np
import pandas as pd
import yfinance as yf
import datetime
import math
import matplotlib.pyplot as plt
from skfolio.datasets import load_sp500_dataset, load_sp500_implied_vol_dataset
from skfolio.preprocessing import prices_to_returns

In [2]:
def calculate_mean_estimator(data):
    data['Natural Return'] = (data['Close'].shift(-1) - data['Close']) / data['Close']
    mu_hat = np.mean(data['Natural Return'])
    return mu_hat

def calculate_unbiased_volatility(data, mu_hat=None):
    if mu_hat is None:
        mu_hat = calculate_mean_estimator(data)
    data['Adjusted Return'] = data['Natural Return'] - mu_hat
    data['Squared Adjusted Return'] = data['Adjusted Return'] ** 2
    realized_volatility_estimate = np.sum(data['Squared Adjusted Return'].dropna(), axis=0) / (len(data) - 1)
    return np.sqrt(realized_volatility_estimate)

def get_log_ratios(price1, price2):
    log_high_low_ratio = np.log(price1 / price2)
    return log_high_low_ratio

def calculate_parkinsons_estimator(data):
    log_high_low_ratio = get_log_ratios(data['High'], data['Low'])
    parkinsons_volatility_estimate = np.sqrt((1 / (4 * np.log(2))) * (log_high_low_ratio ** 2).sum())
    return parkinsons_volatility_estimate

def calculate_garman_klass_estimator(data):
    log_high_low_ratio = get_log_ratios(data['High'], data['Low'])
    first_term = (1 / (2 * len(data))) * (log_high_low_ratio ** 2).sum()

    log_close_open_ratio = get_log_ratios(data['Close'], data['Open'])
    second_term = (2 * np.log(2) - 1) * (log_close_open_ratio ** 2).sum() / len(data)

    garman_klass_estimator = np.sqrt(first_term - second_term)
    return garman_klass_estimator

def get_rolling_window_estimates(data, T):
    classic_measures = []
    park_estimates = []
    garman_estimates = []

    # compute volatility for all windows
    for idx in range(len(data) - T + 1):
        current_window = data.iloc[idx : idx + T].copy()
        classic_measures.append(calculate_unbiased_volatility(current_window))
        park_estimates.append(calculate_parkinsons_estimator(current_window))
        garman_estimates.append(calculate_garman_klass_estimator(current_window))

    return classic_measures, park_estimates, garman_estimates

def plot_rolling_window_estimate(classic_measures, park_estimates, garman_estimates, T): 
    # plot volatility estimates
    plt.plot(classic_measures, label='Classic')
    plt.plot(park_estimates, label='Parkinson')
    plt.plot(garman_estimates, label='Garman-Klass')

    plt.title(f"Volatility Estimates for {T} Days")
    plt.xlabel("Days")
    plt.ylabel("Volatility")
    plt.legend()
    plt.show()

def get_volatility_signature(data, windows):
    classic_measures = []
    park_estimates = []
    garman_estimates = []

    for window in windows:
        classic, park, garman = get_rolling_window_estimates(data, window)
        classic_measures.append(np.mean(classic))
        park_estimates.append(np.mean(park))
        garman_estimates.append(np.mean(garman))

    return classic_measures, park_estimates, garman_estimates

def plot_volatility_signature(windows, avg_classic_list, avg_park_list, avg_garman_list):
    plt.figure(figsize=(10, 6))
    plt.plot(windows, avg_classic_list, marker='o', label='Classic')
    plt.plot(windows, avg_park_list, marker='o', label='Parkinson')
    plt.plot(windows, avg_garman_list, marker='o', label='Garman-Klass')
    plt.xlabel('Window Size (m)')
    plt.ylabel('Average Realized Volatility')
    plt.title('Volatility Signature Plot')
    plt.legend()
    plt.grid(True)
    plt.show()

In [ ]:
ticker = "AAPL"
start_date = "2010-01-01"
end_date = datetime.datetime.now().strftime("%Y-%m-%d")
df = yf.download(ticker, start=start_date, end=end_date)

In [ ]:
garman_klass_volatility = calculate_garman_klass_estimator(df)
print(f"Garman-Klass Volatility Estimate: {garman_klass_volatility}")

parkinsons_volatility = calculate_parkinsons_estimator(df)
print(f"Parkinson's Volatility Estimate: {parkinsons_volatility}")

In [ ]:
# Plot for a window size of 30
window_size = 30
classic_measures, park_estimates, garman_estimates = get_rolling_window_estimates(df, window_size) 
plot_rolling_window_estimate(classic_measures, park_estimates, garman_estimates, window_size)

In [ ]:
# Volatility signature for a range of window sizes
windows = [5, 10, 20, 30, 90, 150, 250]
c_vol_means, p_vol_means, g_vol_means = get_volatility_signature(df, windows)
plot_volatility_signature(windows, c_vol_means, p_vol_means, g_vol_means)

In [ ]:
implied_vol = load_sp500_implied_vol_dataset()

ticker_data = df
ticker_impl_vol = implied_vol[ticker]

# filter the data to start from 2010
ticker_data = ticker_data.loc["2010-01-01":"2022-01-01"]
ticker_impl_vol = ticker_impl_vol.loc["2010-01-01":"2022-01-01"]

classic_measures, park_estimates, garman_estimates = get_rolling_window_estimates(ticker_data, window_size)

# allign implied vol with the other measures
aligned = ticker_data.index[29:]  # T - 1 = 29
realized_vol_df = pd.DataFrame({
    'Classic': classic_measures,
    'Parkinson': park_estimates,
    'Garman-Klass': garman_estimates
}, index=aligned)

common_index = realized_vol_df.index.intersection(ticker_impl_vol.index)
realized_vol_df = realized_vol_df.loc[common_index]
aligned_impl_vol = ticker_impl_vol.loc[common_index]

plt.figure(figsize=(14, 6))
plt.plot(realized_vol_df.index, realized_vol_df['Classic'], label='Realized Vol (Classic)')
plt.plot(realized_vol_df.index, realized_vol_df['Parkinson'], label='Realized Vol (Parkinson)')
plt.plot(realized_vol_df.index, realized_vol_df['Garman-Klass'], label='Realized Vol (Garman-Klass)')
plt.plot(aligned_impl_vol.index, aligned_impl_vol, label='Implied Volatility', linestyle='--', linewidth=2)

plt.title(f'Realized vs Implied Volatility for {ticker} (Rolling Window = {window_size})')
plt.xlabel('Date')
plt.ylabel('Volatility')
plt.legend()
plt.grid(True)
plt.tight_layout()
plt.show()

### Step 4 : Implied Volatility Data and VIX Estimation 

In [1]:
import yfinance as yf 
import datetime 
import utils
import numpy as np 

Set initial variables 

In [2]:
spx_symbol = "^SPX"
today = datetime.datetime.strptime("2025-03-05", "%Y-%m-%d")
end_date = today
start_date = end_date - datetime.timedelta(days=365) 

In [3]:
spx_data = yf.download(spx_symbol, start=start_date, end=end_date)

YF.download() has changed argument auto_adjust default to True


[*********************100%***********************]  1 of 1 completed


In [4]:
last_bus_day = spx_data.index[-1]
print(last_bus_day)

2025-03-04 00:00:00


In [5]:
vix_data = yf.download("^VIX", start=last_bus_day, end=last_bus_day + datetime.timedelta(days=1))  
print(spx_data.tail())
print(vix_data.tail())

[*********************100%***********************]  1 of 1 completed

Price             Close         High          Low         Open      Volume
Ticker             ^SPX         ^SPX         ^SPX         ^SPX        ^SPX
Date                                                                      
2025-02-26  5956.060059  6009.819824  5932.689941  5970.870117  4869580000
2025-02-27  5861.569824  5993.689941  5858.779785  5981.879883  5057680000
2025-02-28  5954.500000  5959.399902  5837.660156  5856.740234  6441140000
2025-03-03  5849.720215  5986.089844  5810.910156  5968.330078  5613850000
2025-03-04  5778.149902  5865.080078  5732.589844  5811.979980  6138110000
Price       Close   High        Low       Open Volume
Ticker       ^VIX   ^VIX       ^VIX       ^VIX   ^VIX
Date                                                 
2025-03-04  23.51  26.35  21.709999  22.959999      0


In [6]:
def find_closest_expiry(spx_symbol, today, days_from_today=30):
    spx_ticker = yf.Ticker(spx_symbol)
    expiry_dates = spx_ticker.options
    if today is None:
        today = datetime.datetime.today()
    expiry_dates_sorted = sorted([datetime.datetime.strptime(expiry, "%Y-%m-%d") for expiry in expiry_dates])
    closest_expiry = None
    min_diff = float('inf')
    for expiry_date in expiry_dates_sorted:
        days_to_expiry = (expiry_date - today).days
        if days_to_expiry >= 0:
            diff = abs(days_to_expiry - days_from_today)
            if diff < min_diff:
                min_diff = diff
                closest_expiry = expiry_date
    return closest_expiry

In [7]:
closest_expiry_date = find_closest_expiry(spx_symbol, today)
closest_expiry_date = closest_expiry_date.strftime('%Y-%m-%d')
print(closest_expiry_date)

2025-04-14


In [8]:
expiry_datetime = datetime.datetime.strptime(closest_expiry_date, "%Y-%m-%d")
days_to_expiry = (expiry_datetime - last_bus_day.to_pydatetime()).days
T = days_to_expiry / 365.0  # Time to expiration in years
print(f"Days to expiration: {days_to_expiry}")
print(f"T (years): {T:.4f}")

Days to expiration: 41
T (years): 0.1123


In [9]:
S0 = float(spx_data['Close'].iloc[-1])  
print(f"Spot Price : {S0}")
F0 = S0 * np.exp(0.02 * (T))
print(f"forward price : {F0}") 

Spot Price : 5778.14990234375
forward price : 5791.145543817414


/var/folders/_1/0wm31tv93_g2x0n62x6b_ht80000gn/T/ipykernel_3511/4272605967.py:1: FutureWarning: Calling float on a single element Series is deprecated and will raise a TypeError in the future. Use float(ser.iloc[0]) instead
  S0 = float(spx_data['Close'].iloc[-1])


In [10]:
closest_expiry_date = find_closest_expiry(spx_symbol, today)
closest_expiry_date = closest_expiry_date.strftime('%Y-%m-%d')
print(closest_expiry_date)

2025-04-14


In [11]:
spx_ticker = yf.Ticker(spx_symbol)
chain = spx_ticker.option_chain(closest_expiry_date)
calls_df = chain.calls.copy()
puts_df = chain.puts.copy()

In [12]:
def calculate_calls_puts_sum(puts_df, calls_df):
    vix_sum = 0
    nputs = len(puts_df)
    for i in range(nputs - 1):
        Kp_i = puts_df.iloc[i]['strike']
        Kp_i_next = puts_df.iloc[i + 1]['strike']
        P_i = puts_df.iloc[i]['lastPrice']
        vix_sum += P_i * (1 / Kp_i - 1 / Kp_i_next)
    
    np_calls = len(calls_df)
    for i in range(1, np_calls):
        Kc_i = calls_df.iloc[i]['strike']
        Kc_i_prev = calls_df.iloc[i - 1]['strike'] if i > 0 else Kc_i  
        C_i = calls_df.iloc[i]['lastPrice']
        vix_sum += C_i * (1 / Kc_i_prev - 1 / Kc_i)
    
    return vix_sum

def VIX_estimator(puts_df, calls_df, F0, r=0.02, tau=41/365):
    vix_sum = calculate_calls_puts_sum(puts_df, calls_df)
    vix_square = (2 * np.exp(r * tau) / tau) * vix_sum
    return np.sqrt(vix_square) * 100  

In [13]:
puts_otm = puts_df[puts_df['strike'] < F0].copy()
calls_otm = calls_df[calls_df['strike'] > F0].copy()

print(VIX_estimator(puts_otm, calls_otm, F0))

          contractSymbol             lastTradeDate  strike  lastPrice   bid  \
230  SPXW250414C05795000 2025-04-14 06:36:05+00:00  5795.0       0.12  0.10   
231  SPXW250414C05800000 2025-04-14 08:14:22+00:00  5800.0       0.20  0.10   
232  SPXW250414C05805000 2025-04-11 19:56:20+00:00  5805.0       0.20  0.10   
233  SPXW250414C05810000 2025-04-14 05:50:36+00:00  5810.0       0.15  0.15   
234  SPXW250414C05815000 2025-04-11 19:47:53+00:00  5815.0       0.20  0.10   
235  SPXW250414C05820000 2025-04-11 20:07:44+00:00  5820.0       0.20  0.10   
236  SPXW250414C05825000 2025-04-14 06:29:40+00:00  5825.0       0.10  0.10   
237  SPXW250414C05830000 2025-04-14 07:38:38+00:00  5830.0       0.15  0.10   
238  SPXW250414C05840000 2025-04-14 05:44:55+00:00  5840.0       0.05  0.05   
239  SPXW250414C05850000 2025-04-14 07:57:34+00:00  5850.0       0.15  0.05   
240  SPXW250414C05860000 2025-04-11 20:11:53+00:00  5860.0       0.20  0.05   
241  SPXW250414C05870000 2025-04-11 20:13:09+00:00  

In [14]:
vix_data = yf.download("^VIX", start=last_bus_day, end=last_bus_day + datetime.timedelta(days=1))
print(vix_data['Close'])

[*********************100%***********************]  1 of 1 completed

Ticker       ^VIX
Date             
2025-03-04  23.51
